In [1]:
#imports and configurations
import pandas as pd
import numpy as np
import os
import re

from pathlib import Path

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
#loading the datasets
DATA_DIR = Path(r"C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data")

dataset_files = {
    "CEAS_08.csv": DATA_DIR / "CEAS_08.csv",
    "Enron.csv": DATA_DIR / "Enron.csv",
    "Ling.csv": DATA_DIR / "Ling.csv",
    "Nazario.csv": DATA_DIR / "Nazario.csv",
    "Nigerian_Fraud.csv": DATA_DIR / "Nigerian_Fraud.csv",
    "phishing_email.csv": DATA_DIR / "phishing_email.csv",
    "SpamAssasin.csv": DATA_DIR / "SpamAssasin.csv"
}

datasets = {}

for name, path in dataset_files.items():
    datasets[name] = pd.read_csv(path)
    print(f"{name}: {datasets[name].shape}")

CEAS_08.csv: (39154, 7)
Enron.csv: (29767, 3)
Ling.csv: (2859, 3)
Nazario.csv: (1565, 7)
Nigerian_Fraud.csv: (3332, 7)
phishing_email.csv: (82486, 2)
SpamAssasin.csv: (5809, 7)


In [3]:
#creating a text column
def create_text_column(df):
    
    df = df.copy()

    if "text_combined" in df.columns:
        df["text"] = df["text_combined"].fillna("").astype(str)

    else:
        subject = df["subject"].fillna("").astype(str) if "subject" in df.columns else ""
        body = df["body"].fillna("").astype(str) if "body" in df.columns else ""

        df["text"] = subject + " " + body

    return df

In [4]:
for name in datasets:
    datasets[name] = create_text_column(datasets[name])

print("Common text column created.")

Common text column created.


In [5]:
#basic text cleaning
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text

In [6]:
#applying the text cleaning
for name in datasets:
    datasets[name]["text"] = datasets[name]["text"].apply(clean_text)

print("Text normalization completed.")

Text normalization completed.


In [7]:
#removing the empty messages
for name in datasets:
    before = len(datasets[name])

    datasets[name] = datasets[name][
        datasets[name]["text"].str.len() > 0
    ].copy()

    after = len(datasets[name])

    print(f"{name}: removed {before - after} empty messages")

CEAS_08.csv: removed 0 empty messages
Enron.csv: removed 0 empty messages
Ling.csv: removed 0 empty messages
Nazario.csv: removed 0 empty messages
Nigerian_Fraud.csv: removed 0 empty messages
phishing_email.csv: removed 1 empty messages
SpamAssasin.csv: removed 0 empty messages


In [8]:
#removing extreamly short messages
MIN_LENGTH = 10

for name in datasets:
    before = len(datasets[name])

    datasets[name] = datasets[name][
        datasets[name]["text"].str.len() >= MIN_LENGTH
    ].copy()

    after = len(datasets[name])

    print(f"{name}: removed {before - after} very short messages")

CEAS_08.csv: removed 0 very short messages
Enron.csv: removed 3 very short messages
Ling.csv: removed 0 very short messages
Nazario.csv: removed 0 very short messages
Nigerian_Fraud.csv: removed 0 very short messages
phishing_email.csv: removed 5 very short messages
SpamAssasin.csv: removed 0 very short messages


In [9]:
#removing duplicate messages within dataset
for name in datasets:
    before = len(datasets[name])

    datasets[name] = datasets[name].drop_duplicates(
        subset=["text"]
    ).copy()

    after = len(datasets[name])

    print(f"{name}: removed {before - after} duplicates")

CEAS_08.csv: removed 191 duplicates
Enron.csv: removed 344 duplicates
Ling.csv: removed 0 duplicates
Nazario.csv: removed 13 duplicates
Nigerian_Fraud.csv: removed 39 duplicates
phishing_email.csv: removed 408 duplicates
SpamAssasin.csv: removed 7 duplicates


In [10]:
#limittning extreme message length
MAX_LENGTH = 10000

for name in datasets:
    datasets[name]["text"] = datasets[name]["text"].str.slice(
        0, MAX_LENGTH
    )

print("Maximum text length limited to:", MAX_LENGTH)

Maximum text length limited to: 10000


In [11]:
#creating a dataset summary
summary = []

for name, df in datasets.items():

    summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "avg_length": round(df["text"].str.len().mean(), 2),
        "median_length": round(df["text"].str.len().median(), 2),
        "duplicates": df["text"].duplicated().sum(),
        "empty": (df["text"].str.len() == 0).sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

,dataset,rows,columns,avg_length,median_length,duplicates,empty
0,CEAS_08.csv,38963,8,1363.70,578.0,4,0
1,Enron.csv,29420,4,1332.35,727.0,1,0
2,Ling.csv,2859,4,2985.76,2055.0,0,0
3,Nazario.csv,1552,8,1095.77,730.0,0,0
4,Nigerian_Fraud.csv,3293,8,2578.40,2533.0,0,0
5,phishing_email.csv,82072,3,1117.69,558.0,5,0
6,SpamAssasin.csv,5802,8,1845.35,1087.0,5,0


In [12]:
#saving the cleaned individual data
CLEAN_DIR = DATA_DIR / "cleaned"

CLEAN_DIR.mkdir(exist_ok=True)

for name, df in datasets.items():

    output_path = CLEAN_DIR / name

    df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data\cleaned\CEAS_08.csv
Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data\cleaned\Enron.csv
Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data\cleaned\Ling.csv
Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data\cleaned\Nazario.csv
Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data\cleaned\Nigerian_Fraud.csv
Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data\cleaned\phishing_email.csv
Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data\cleaned\SpamAssasin.csv


In [13]:
#creating a combined working dataset
# Datasets used for the first CyberGuard text detector
general_datasets = [
    "CEAS_08.csv",
    "Enron.csv",
    "Ling.csv",
    "SpamAssasin.csv",
    "phishing_email.csv"
]

phishing_datasets = [
    "Nazario.csv",
    "Nigerian_Fraud.csv"
]

combined = []

# General suspicious/legitimate datasets
for name in general_datasets:
    df = datasets[name].copy()

    df["source_dataset"] = name
    df["source_label"] = df["label"]

    # Preserve original labels for analysis
    # CyberGuard working label:
    # 0 = legitimate
    # 1 = suspicious
    df["cyberguard_label"] = df["label"]

    combined.append(
        df[["text", "cyberguard_label", "source_dataset", "source_label"]]
    )


# Strong phishing/fraud datasets
for name in phishing_datasets:
    df = datasets[name].copy()

    df["source_dataset"] = name
    df["source_label"] = df["label"]

    # These datasets contain positive phishing/fraud examples
    df["cyberguard_label"] = 1

    combined.append(
        df[["text", "cyberguard_label", "source_dataset", "source_label"]]
    )


combined_df = pd.concat(combined, ignore_index=True)

print("Combined shape:", combined_df.shape)
print("\nCyberGuard labels:")
print(combined_df["cyberguard_label"].value_counts())

print("\nSources:")
print(combined_df["source_dataset"].value_counts())

Combined shape: (163961, 4)

CyberGuard labels:
cyberguard_label
1    85474
0    78487
Name: count, dtype: int64

Sources:
source_dataset
phishing_email.csv    82072
CEAS_08.csv           38963
Enron.csv             29420
SpamAssasin.csv        5802
Nigerian_Fraud.csv     3293
Ling.csv               2859
Nazario.csv            1552
Name: count, dtype: int64


In [14]:
#checking the resulting class distribution
print(
    combined_df["cyberguard_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

cyberguard_label
1    52.13
0    47.87
Name: proportion, dtype: float64


In [15]:
print(
    pd.crosstab(
        combined_df["source_dataset"],
        combined_df["cyberguard_label"]
    )
)

cyberguard_label        0      1
source_dataset                  
CEAS_08.csv         17289  21674
Enron.csv           15474  13946
Ling.csv             2401    458
Nazario.csv             0   1552
Nigerian_Fraud.csv      0   3293
SpamAssasin.csv      4090   1712
phishing_email.csv  39233  42839


In [16]:
#Checking for cross-source duplicates again
duplicate_mask = combined_df["text"].duplicated(keep=False)

print("Cross-source duplicate rows:", duplicate_mask.sum())

duplicates = combined_df[duplicate_mask].sort_values("text")

duplicates[
    ["text", "cyberguard_label", "source_dataset"]
].head(20)

Cross-source duplicate rows: 72


,text,cyberguard_label,source_dataset
2819,Choose Your Direction for RIAs at AJAXWorld 20...,0,CEAS_08.csv
3195,Choose Your Direction for RIAs at AJAXWorld 20...,0,CEAS_08.csv
18756,Find Out How To Follow Fortune 500's RIA Pione...,0,CEAS_08.csv
20737,Find Out How To Follow Fortune 500's RIA Pione...,0,CEAS_08.csv
31958,MSDN Flash - Heroes Happen Here: the 2008 Laun...,0,CEAS_08.csv
31960,MSDN Flash - Heroes Happen Here: the 2008 Laun...,0,CEAS_08.csv
75568,Newsletter Coach'Invest - Septembre 2002 NEWSL...,1,SpamAssasin.csv
75567,Newsletter Coach'Invest - Septembre 2002 NEWSL...,1,SpamAssasin.csv
76756,"The Government Grants You $25,000! # ** Free P...",1,SpamAssasin.csv
76729,"The Government Grants You $25,000! # ** Free P...",1,SpamAssasin.csv


In [17]:
#Removing cross-source duplicates
before = len(combined_df)

combined_df = combined_df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

after = len(combined_df)

print("Removed:", before - after)
print("Final working samples:", after)

Removed: 37
Final working samples: 163924


In [18]:
#saving the combined dataset
output_path = CLEAN_DIR / "cyberguard_text_dataset.csv"

combined_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\phishing_detection\data\cleaned\cyberguard_text_dataset.csv
